# TP1 avec SQLite

Version SQLite du TP1 (originalement écrit pour PostgreSQL). On utilise le même jeu de données Northwind, mais :
- pas de serveur, pas de credentials, juste un fichier
- on charge le dump `data/northwind.sql` (généré par `pg_dump`) en l'adaptant à la volée
- on affiche les résultats sous forme de DataFrame pandas pour le confort


## 0. Setup

### Chargement de Northwind dans SQLite

Le dump PostgreSQL est presque compatible. Quatre petites adaptations suffisent (voir `course/setup/setup_sqlite.ipynb` pour le détail).

In [ ]:
import os
import re
import sqlite3

DUMP = "../../../data/northwind.sql"
DB = ":memory:"   # ou "northwind.sqlite" pour persister

with open(DUMP) as f:
    sql = f.read()

sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
sql = sql.replace("bytea", "BLOB")
sql = sql.replace(r"'\x'", "NULL")

conn = sqlite3.connect(DB)
conn.executescript(sql)
conn.commit()
print("Northwind chargé. SQLite", sqlite3.sqlite_version)

### Helper d'affichage

`q(sql)` exécute une requête et renvoie un DataFrame pandas (rendu propre dans Jupyter).

In [ ]:
import pandas as pd

def q(sql):
    return pd.read_sql_query(sql, conn)

q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

## 1. Rapatrier les données d'une table

- `SELECT` : rapatrier les données d'une seule table
- Alias de colonnes : assigner des noms temporaires
- `ORDER BY` : trier les résultats
- `SELECT DISTINCT` : retirer les doublons


### SELECT

Une seule colonne :

In [ ]:
q("SELECT contact_name FROM customers LIMIT 5")

Plusieurs colonnes :

In [ ]:
q("SELECT contact_name, contact_title, city FROM customers LIMIT 5")

Toutes les colonnes :

In [ ]:
q("SELECT * FROM customers LIMIT 3")

Avec une expression (concaténation : `||` comme en PostgreSQL) :

In [ ]:
q("SELECT contact_name || ', ' || city AS qui_ou FROM customers LIMIT 5")

### Alias de colonne (AS)

L'alias rend le nom de la colonne dans le résultat plus parlant.

In [ ]:
q("""
SELECT contact_name || ' ' || contact_title AS whoami
FROM customers
LIMIT 5
""")

### ORDER BY

Tri croissant (par défaut) ou décroissant, sur une ou plusieurs colonnes.

In [ ]:
q("""
SELECT ship_address, ship_name
FROM orders
ORDER BY shipped_date ASC
LIMIT 5
""")

In [ ]:
q("""
SELECT ship_address, ship_name
FROM orders
ORDER BY order_date ASC, shipped_date DESC
LIMIT 5
""")

Tri sur une expression :

In [ ]:
q("""
SELECT ship_address, LENGTH(ship_address) AS len
FROM orders
ORDER BY len DESC
LIMIT 5
""")

**NULL et tri** : SQLite supporte `NULLS FIRST` / `NULLS LAST` depuis la version 3.30. Sinon, par défaut les `NULL` sont en premier en ASC, en dernier en DESC.

In [ ]:
q("""
SELECT order_id, ship_region
FROM orders
ORDER BY ship_region NULLS LAST
LIMIT 5
""")

### DISTINCT

In [ ]:
q("SELECT DISTINCT ship_name FROM orders LIMIT 10")

In [ ]:
q("SELECT DISTINCT ship_country, ship_city FROM orders LIMIT 10")

**Différence avec PostgreSQL** : SQLite n'a pas `DISTINCT ON (col)`. Contournement avec `ROW_NUMBER()` :

```sql
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY ship_name ORDER BY order_date) AS rn
  FROM orders
) WHERE rn = 1;
```

## 2. Filtrer les données d'une table

- `WHERE` : filtrer les lignes selon une condition
- `LIMIT` (et `OFFSET`) : sous‑ensemble de résultats
- `IS NULL` / `IS NOT NULL` : tester la nullité


### WHERE

Égalité :

In [ ]:
q("SELECT * FROM orders WHERE ship_country = 'France' LIMIT 5")

AND :

In [ ]:
q("SELECT * FROM orders WHERE ship_country = 'USA' AND freight > 100 LIMIT 5")

OR :

In [ ]:
q("SELECT * FROM orders WHERE ship_city = 'Paris' OR ship_city = 'Lyon' LIMIT 5")

IN :

In [ ]:
q("SELECT * FROM orders WHERE ship_country IN ('USA', 'France', 'Germany') LIMIT 5")

LIKE :

**Attention** : sous SQLite, `LIKE` est insensible à la casse pour les caractères ASCII par défaut (l'inverse de PostgreSQL). Pour forcer la sensibilité : `PRAGMA case_sensitive_like = ON;` ou utiliser `GLOB` (qui est case‑sensitive).

In [ ]:
q("SELECT * FROM orders WHERE ship_name LIKE 'C%' LIMIT 5")

BETWEEN :

In [ ]:
q("SELECT * FROM orders WHERE order_date BETWEEN '1997-01-01' AND '1997-01-31' LIMIT 5")

Différent : `<>` ou `!=`

In [ ]:
q("SELECT * FROM orders WHERE ship_country <> 'USA' LIMIT 5")

### LIMIT

Limiter le nombre de lignes, éventuellement avec un décalage (`OFFSET`).

In [ ]:
q("SELECT * FROM orders ORDER BY order_date LIMIT 5")

In [ ]:
q("SELECT * FROM orders ORDER BY order_date LIMIT 4 OFFSET 3")

### IS NULL / IS NOT NULL

`NULL` représente une information manquante. On ne peut pas le tester avec `=` (qui renvoie toujours faux), il faut `IS NULL` ou `IS NOT NULL`.

In [ ]:
q("SELECT * FROM orders WHERE ship_region IS NULL LIMIT 5")

In [ ]:
q("SELECT * FROM orders WHERE ship_region IS NOT NULL LIMIT 5")

## 3. Exercices

Mêmes énoncés que `tp1/2_exercices.ipynb`. Les corrigés sont juste en dessous.


### Exercice 1 : sélection avec conditions multiples

Afficher tous les `order_id` et `ship_name` pour les commandes expédiées en Suisse (`ship_country = 'Switzerland'`) ou en Argentine (`ship_country = 'Argentina'`), avec un `freight` supérieur à 20.

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("""
SELECT order_id, ship_name
FROM orders
WHERE (ship_country = 'Switzerland' OR ship_country = 'Argentina')
  AND freight > 20
""")

### Exercice 2 : tri et limite sur sélection filtrée

Trouver les 5 commandes ayant le plus haut `freight`, expédiées en 1997.

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("""
SELECT *
FROM orders
WHERE order_date BETWEEN '1997-01-01' AND '1997-12-31'
ORDER BY freight DESC
LIMIT 5
""")

### Exercice 3 : utilisation de DISTINCT

Lister les combinaisons uniques de `ship_country` et `ship_city` pour toutes les commandes.

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("SELECT DISTINCT ship_country, ship_city FROM orders")

### Exercice 4 : combiner WHERE, ORDER BY et LIMIT

Afficher les 3 premières commandes (basées sur `order_date`) pour les clients au Brésil (`ship_country = 'Brazil'`).

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("""
SELECT *
FROM orders
WHERE ship_country = 'Brazil'
ORDER BY order_date
LIMIT 3
""")

### Exercice 5 : gestion des NULL avec tri et filtrage

Sélectionner toutes les commandes où `ship_region` est `NULL`, triées par `shipped_date` en ordre décroissant.

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("""
SELECT *
FROM orders
WHERE ship_region IS NULL
ORDER BY shipped_date DESC
""")

### Exercice 6 : utilisation de l'opérateur LIKE

Trouver les commandes dont le nom du destinataire (`ship_name`) commence par 'S'.

Rappel : sous SQLite, `LIKE` est insensible à la casse pour l'ASCII. `'S%'` et `'s%'` donnent le même résultat. Pour s'aligner sur PostgreSQL (case‑sensitive), utiliser `GLOB`.

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("SELECT * FROM orders WHERE ship_name LIKE 'S%'")

Équivalent strictement case‑sensitive :

In [ ]:
q("SELECT * FROM orders WHERE ship_name GLOB 'S*'")

### Exercice 7 : plusieurs conditions avec AND, OR

Sélectionner les commandes expédiées en 'UK' ou 'USA' avec un `freight` supérieur à 50, et passées avant l'année 1998.

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("""
SELECT *
FROM orders
WHERE (ship_country = 'UK' OR ship_country = 'USA')
  AND freight > 50
  AND order_date < '1998-01-01'
""")

### Exercice 8 : requête à plusieurs clauses

Afficher les `order_id`, la date de commande (`order_date`), et le nom du destinataire (`ship_name`) pour les 10 premières commandes passées après le 1er janvier 1997, où la région de livraison (`ship_region`) est inconnue (`NULL`), et le pays de livraison (`ship_country`) n'est ni 'USA' ni 'UK'. Utiliser un alias pour renommer `order_date` en `DateOfOrder`.

In [ ]:
# à vous


**Corrigé**

In [ ]:
q("""
SELECT order_id, order_date AS DateOfOrder, ship_name
FROM orders
WHERE order_date > '1997-01-01'
  AND ship_region IS NULL
  AND ship_country NOT IN ('USA', 'UK')
ORDER BY order_date ASC, freight DESC
LIMIT 10
""")

## Fermeture

In [ ]:
conn.close()